# REPATH 03 — Calibrate the confidence margin

This notebook selects the confidence-margin threshold from pooled development out-of-fold scores and applies the frozen threshold to the held-out test scores. It does not train or tune the ranking model.

The notebook reads the calibration inputs under `artifacts/repath/`, reads `models/config_base.pkl`, writes calibration tables under `results/calibration/`, and writes the calibrated runtime configuration to `models/config.pkl`.


## 1. Import dependencies


In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "configs" / "site_split.yaml").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from the repository root or the notebooks directory."
    )


print("Calibration imports ready.")


## 2. Configure repository inputs and outputs


In [ ]:
REPO_ROOT = find_repository_root()
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "repath"
MODEL_ROOT = REPO_ROOT / "models"
RESULT_ROOT = REPO_ROOT / "results" / "calibration"

QUERY_MANIFEST_CSV = ARTIFACT_ROOT / "repath_query_manifest.csv"
DEVELOPMENT_OOF_SCORES_CSV = ARTIFACT_ROOT / "development_oof_candidate_scores.csv"
TEST_SCORES_CSV = ARTIFACT_ROOT / "test_candidate_scores.csv"
BASE_CONFIG_PATH = MODEL_ROOT / "config_base.pkl"
FINAL_CONFIG_PATH = MODEL_ROOT / "config.pkl"

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_PRECISION = 0.99
MIN_END_TO_END_COVERAGE = 0.80
TAU_MIN = 0.00
TAU_MAX = 1.00
TAU_STEP = 0.05

print("Repository root    :", REPO_ROOT)
print("Manifest           :", QUERY_MANIFEST_CSV)
print("Development scores :", DEVELOPMENT_OOF_SCORES_CSV)
print("Test scores        :", TEST_SCORES_CSV)
print("Base config        :", BASE_CONFIG_PATH)
print("Final config       :", FINAL_CONFIG_PATH)
print("Calibration reports:", RESULT_ROOT)


## 3. Load and validate calibration inputs


In [ ]:
for required_path in [
    QUERY_MANIFEST_CSV,
    DEVELOPMENT_OOF_SCORES_CSV,
    TEST_SCORES_CSV,
    BASE_CONFIG_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {required_path}"
        )

manifest_df = pd.read_csv(
    QUERY_MANIFEST_CSV,
    keep_default_na=False,
)
development_scored = pd.read_csv(
    DEVELOPMENT_OOF_SCORES_CSV,
    keep_default_na=False,
)
test_scored = pd.read_csv(
    TEST_SCORES_CSV,
    keep_default_na=False,
)
base_config = joblib.load(
    BASE_CONFIG_PATH
)

required_manifest_columns = {
    "qid",
    "site",
    "split",
}
required_score_columns = {
    "qid",
    "label",
    "score",
}

missing_manifest = (
    required_manifest_columns
    - set(manifest_df.columns)
)
if missing_manifest:
    raise ValueError(
        f"Manifest missing columns: "
        f"{sorted(missing_manifest)}"
    )

for name, scored in [
    ("development", development_scored),
    ("test", test_scored),
]:
    missing = (
        required_score_columns
        - set(scored.columns)
    )
    if missing:
        raise ValueError(
            f"{name} scores missing columns: "
            f"{sorted(missing)}"
        )

    scored["qid"] = scored["qid"].astype(str)
    scored["label"] = pd.to_numeric(
        scored["label"],
        errors="raise",
    ).astype(int)
    scored["score"] = pd.to_numeric(
        scored["score"],
        errors="raise",
    ).astype(float)

    positive_counts = (
        scored.groupby("qid")["label"].sum()
    )
    if not positive_counts.eq(1).all():
        bad = positive_counts[
            ~positive_counts.eq(1)
        ]
        raise ValueError(
            f"{name}: each retrievable query must "
            f"have exactly one positive. "
            f"Examples: {bad.head().to_dict()}"
        )

manifest_df["qid"] = (
    manifest_df["qid"].astype(str)
)

development_framework_qids = set(
    manifest_df.loc[
        manifest_df["split"] == "development",
        "qid",
    ]
)
test_framework_qids = set(
    manifest_df.loc[
        manifest_df["split"] == "test",
        "qid",
    ]
)

development_score_qids = set(
    development_scored["qid"]
)
test_score_qids = set(
    test_scored["qid"]
)

if not development_score_qids.issubset(
    development_framework_qids
):
    raise ValueError(
        "Development scores contain unknown queries"
    )

if not test_score_qids.issubset(
    test_framework_qids
):
    raise ValueError(
        "Test scores contain unknown queries"
    )

print(
    "Development framework/retrievable:",
    len(development_framework_qids),
    "/",
    len(development_score_qids),
)
print(
    "Test framework/retrievable:",
    len(test_framework_qids),
    "/",
    len(test_score_qids),
)


## 4. Build the development OOF precision–coverage curve

Threshold selection maximises **end-to-end coverage** subject to the requested precision constraint. Retrieval failures are included in the end-to-end denominator.


In [ ]:
def score_margin_table(scored):
    ranked = scored[
        ["qid", "label", "score"]
    ].copy()

    ranked = ranked.sort_values(
        ["qid", "score"],
        ascending=[True, False],
        kind="mergesort",
    )

    top1 = (
        ranked.groupby("qid", sort=False)
        .nth(0)
        .reset_index()
        .rename(
            columns={
                "score": "s1",
                "label": "label1",
            }
        )
    )

    top2 = (
        ranked.groupby("qid", sort=False)
        .nth(1)
        .reset_index()
        .rename(columns={"score": "s2"})
    )

    margins = top1.merge(
        top2[["qid", "s2"]],
        on="qid",
        how="left",
    )

    margins["s2"] = margins["s2"].fillna(
        margins["s1"]
    )
    margins["margin"] = (
        margins["s1"] - margins["s2"]
    )

    return margins


def tau_curve(scored, split, taus):
    margins = score_margin_table(scored)

    framework_queries = int(
        manifest_df.loc[
            manifest_df["split"] == split,
            "qid",
        ].nunique()
    )
    retrievable_queries = int(
        len(margins)
    )

    rows = []

    for tau in taus:
        accepted = margins[
            margins["margin"] >= tau
        ]

        accepted_count = int(
            len(accepted)
        )
        correct = int(
            accepted["label1"].sum()
        )

        precision = (
            float(
                accepted["label1"].mean()
            )
            if accepted_count
            else 1.0
        )

        conditional_coverage = (
            accepted_count / retrievable_queries
            if retrievable_queries
            else 0.0
        )
        end_to_end_coverage = (
            accepted_count / framework_queries
            if framework_queries
            else 0.0
        )

        conditional_recall = (
            correct / retrievable_queries
            if retrievable_queries
            else 0.0
        )
        end_to_end_recall = (
            correct / framework_queries
            if framework_queries
            else 0.0
        )

        conditional_f1 = (
            2
            * precision
            * conditional_recall
            / (
                precision
                + conditional_recall
            )
            if (
                precision
                + conditional_recall
            )
            else 0.0
        )

        end_to_end_f1 = (
            2
            * precision
            * end_to_end_recall
            / (
                precision
                + end_to_end_recall
            )
            if (
                precision
                + end_to_end_recall
            )
            else 0.0
        )

        rows.append({
            "tau_margin": float(tau),
            "accepted": accepted_count,
            "correct": correct,
            "framework_queries": (
                framework_queries
            ),
            "retrievable_queries": (
                retrievable_queries
            ),
            "retrieval_failures": (
                framework_queries
                - retrievable_queries
            ),
            "candidate_recall": (
                retrievable_queries
                / framework_queries
                if framework_queries
                else 0.0
            ),
            "precision": precision,
            "conditional_coverage": (
                conditional_coverage
            ),
            "end_to_end_coverage": (
                end_to_end_coverage
            ),
            "conditional_recall": (
                conditional_recall
            ),
            "end_to_end_recall": (
                end_to_end_recall
            ),
            "conditional_f1": (
                conditional_f1
            ),
            "end_to_end_f1": (
                end_to_end_f1
            ),
        })

    return pd.DataFrame(rows)


taus = np.round(
    np.arange(
        TAU_MIN,
        TAU_MAX + TAU_STEP / 2,
        TAU_STEP,
    ),
    10,
)

development_tau_curve = tau_curve(
    development_scored,
    "development",
    taus,
)

eligible = development_tau_curve[
    (
        development_tau_curve["precision"]
        >= TARGET_PRECISION
    )
    & (
        development_tau_curve[
            "end_to_end_coverage"
        ]
        >= MIN_END_TO_END_COVERAGE
    )
].copy()

precision_constraint_met = (
    not eligible.empty
)

if eligible.empty:
    warnings.warn(
        "No threshold met the requested precision "
        "and minimum end-to-end coverage. "
        "Selecting the highest-precision feasible "
        "development threshold instead."
    )

    eligible = development_tau_curve[
        development_tau_curve[
            "end_to_end_coverage"
        ]
        >= MIN_END_TO_END_COVERAGE
    ].copy()

if eligible.empty:
    raise RuntimeError(
        "No threshold satisfies the minimum "
        "end-to-end coverage"
    )

if precision_constraint_met:
    chosen = (
        eligible.sort_values(
            [
                "end_to_end_coverage",
                "precision",
                "end_to_end_f1",
            ],
            ascending=[
                False,
                False,
                False,
            ],
            kind="mergesort",
        )
        .iloc[0]
    )
else:
    chosen = (
        eligible.sort_values(
            [
                "precision",
                "end_to_end_coverage",
                "end_to_end_f1",
            ],
            ascending=[
                False,
                False,
                False,
            ],
            kind="mergesort",
        )
        .iloc[0]
    )

tau_margin = float(
    chosen["tau_margin"]
)

print(
    "Chosen development OOF threshold:"
)
display(chosen.to_frame().T)

print(
    "Requested precision constraint met:",
    precision_constraint_met,
)

display(development_tau_curve)


## 5. Apply the frozen threshold to development OOF and held-out test scores


In [ ]:
def evaluate_tau(
    scored,
    split,
    tau,
    name,
):
    result = tau_curve(
        scored,
        split,
        [tau],
    ).iloc[0].to_dict()

    result["name"] = name
    return result


development_tau_result = evaluate_tau(
    development_scored,
    "development",
    tau_margin,
    (
        "Development four-fold OOF "
        "threshold performance"
    ),
)

test_tau_result = evaluate_tau(
    test_scored,
    "test",
    tau_margin,
    "Held-out test threshold performance",
)

tau_summary_df = pd.DataFrame([
    development_tau_result,
    test_tau_result,
])

display(tau_summary_df)


## 6. Save calibration tables and the final runtime configuration


In [ ]:
calibration_table = development_tau_curve[
    [
        "tau_margin",
        "accepted",
        "correct",
        "precision",
        "conditional_coverage",
        "end_to_end_coverage",
        "conditional_recall",
        "end_to_end_recall",
        "conditional_f1",
        "end_to_end_f1",
    ]
].copy()
calibration_table.insert(
    1,
    "selected",
    np.isclose(calibration_table["tau_margin"], tau_margin),
)

development_tau_curve.to_csv(
    RESULT_ROOT / "tau_margin_development_oof_curve.csv",
    index=False,
)
tau_summary_df.to_csv(
    RESULT_ROOT / "tau_margin_summary.csv",
    index=False,
)
calibration_table.to_csv(
    RESULT_ROOT / "tau_margin_development_oof_table.csv",
    index=False,
)

display(calibration_table)

final_config = dict(base_config)
final_config.update({
    "tau_margin": float(tau_margin),
    "target_precision": float(TARGET_PRECISION),
    "minimum_end_to_end_coverage": float(MIN_END_TO_END_COVERAGE),
    "precision_constraint_met": bool(precision_constraint_met),
    "development_tau_metrics": development_tau_result,
    "test_tau_metrics": test_tau_result,
    "calibration_protocol": "pooled_four_fold_development_oof",
})

joblib.dump(final_config, FINAL_CONFIG_PATH)

print("Saved:")
print(" ", RESULT_ROOT / "tau_margin_development_oof_curve.csv")
print(" ", RESULT_ROOT / "tau_margin_development_oof_table.csv")
print(" ", RESULT_ROOT / "tau_margin_summary.csv")
print(" ", FINAL_CONFIG_PATH)
print("Selected tau margin:", f"{tau_margin:.6f}")


## Reference output

The bundled calibration inputs select `tau = 0.40` under the 99% development precision target and the 80% minimum end-to-end coverage requirement.

| Split | Accepted | Correct | Precision | Conditional coverage | End-to-end coverage | Conditional recall | End-to-end recall | Conditional F1 | End-to-end F1 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Development four-fold OOF | 764 | 757 | 99.08% | 97.82% | 97.70% | 96.93% | 96.80% | 97.99% | 97.93% |
| Held-out test | 303 | 298 | 98.35% | 96.50% | 95.28% | 94.90% | 93.71% | 96.60% | 95.97% |

The complete threshold sweep is displayed as a table and saved under `results/calibration/`. No calibration figure is generated.
